In [3]:
# numpy는 배열 계산을 쉽게 하기 위해 사용
import numpy as np
from math import e

In [4]:
# 1. 입력값 X와 정답 y 준비

# X: 입력값으로, 사람의 키(cm)를 사용
X = np.array([160, 170, 180, 190])

# y: 정답값으로, 0은 농구선수 아님, 1은 농구선수임을 나타냄
y = np.array([0, 0, 1, 1])

print('입력값 X:', X)
print('입력값 y:', y)

입력값 X: [160 170 180 190]
입력값 y: [0 0 1 1]


In [5]:
# 2. 입력값 정규화

# 평균과 표준편차를 계산
# 평균은 데이터의 중심, 표준편차는 데이터가 평균에서 얼마나 퍼져 있는지를 나타냄
X_mean = np.mean(X)
X_std = np.std(X)

# 정규화 공식: (원본값 - 평균) / 표준편차
# 입력값의 범위를 비슷하게 맞추면 학습이 더 안정적으로 진행됨
X_norm = (X - X_mean) / X_std

print('입력값 평균 X_mean:', X_mean)
print('입력값 표준편차 X_std:', X_std)
print('정규화된 입력값 X_norm:', X_norm)

입력값 평균 X_mean: 175.0
입력값 표준편차 X_std: 11.180339887498949
정규화된 입력값 X_norm: [-1.34164079 -0.4472136   0.4472136   1.34164079]


In [6]:
# 3. 가중치 a와 편향 b 초기값 설정

a = 0.1 # 원래 키(cm)가 아니라, 정규화된 입력값 X_norm에 곱해지는 값
b = 0.0

In [7]:
# 4. 시그모이드 함수 정의
# 시그모이드 함수는 선형 계산값 H(x)를 0~1 사이의 확률값 z로 변환

def sigmoid(H):
    return 1 / (1 + e**(-H))

In [8]:
# 5. 학습 전 예측 확인

# H(x) = ax + b
# H(x)는 퍼셉트론의 선형 계산값
H = a * X_norm + b

# z = sigmoid(H)
# z는 농구선수일 예측 확률
z = sigmoid(H)

print('\n학습 전 선형 계산값 H(x):', H)
print('학습 전 시그모이드 예측 확률 z:', z)


학습 전 선형 계산값 H(x): [-0.13416408 -0.04472136  0.04472136  0.13416408]
학습 전 시그모이드 예측 확률 z: [0.4665092  0.48882152 0.51117848 0.5334908 ]


In [9]:
# 6. 학습 전 비용(Cost) 계산

# Binary Cross Entropy 비용 함수
# 실제값 y가 1이면 z가 1에 가까울수록 비용이 작아지고,
# 실제값 y가 0이면 z가 0에 가까울수록 비용이 작아짐

# Cost = -{y * log(z) + (1-y) * log(1-z)}

# log(0)을 방지하기 위해 z가 정확히 0 또는 1이 되지 않도록 제한
epsilon = 1e-7
z_safe = np.clip(z, epsilon, 1 - epsilon)

costs = -(y * np.log(z_safe) + (1-y) * np.log(1-z_safe))
mean_cost = np.mean(costs)

print('학습 전 각 샘플의 비용(Cost):', costs)
print('학습 전 평균 비용(Cost):', mean_cost)

학습 전 각 샘플의 비용(Cost): [0.62831346 0.67103648 0.67103648 0.62831346]
학습 전 평균 비용(Cost): 0.6496749678557904


In [10]:
# 7. 학습 설정

# learning_rate는 한 번에 얼마나 크게 이동할지를 정하는 학습률
learning_rate = 0.1

# epochs는 전체 데이터를 몇 번 반복해서 학습할지 정하는 값
epochs = 1000

In [11]:
# 경사 하강법으로 학습

for epoch in range(epochs):
    
    # 1단계: 선형 계산 H(x) = ax + b
    H = a * X_norm + b
    print(f'H={H}')
    
    # 2단계: 시그모이드 함수를 적용해 예측 확률 z 계산
    z = sigmoid(H)
    print(f'z={z}')
    
    # 3단계: log(0) 방지
    z_safe = np.clip(z, epsilon, 1-epsilon)
    
    # 4단계: 기울기 계산
    # BCE와 sigmoid를 함께 미분하면 z - y가 등장
    # z는 예측 확률, y는 실제 정답
    grad_a = np.mean((z-y) * X_norm) # Cost를 a로 미분한 값 -> a를 바꾸면 Cost가 어떻게 변하는지를 알려줌
    print(f'grad_a={grad_a}') 
    
    grad_b = np.mean((z-y)) # Cost를 b로 미분한 값 -> b를 바꾸면 Cost가 어떻게 변하는지를 알려줌
    print(f'grad_b={grad_b}')
    
    # 5단계: 경사 하강법으로 a, b 업데이트
    # 새로운 값 = 기존 값 - 학습률 * 기울기
    a = a - learning_rate * grad_a
    b = b - learning_rate * grad_b
    print(f'update a = {a}, update b = {b}')
    
    # z가 epsilon보다 작으면 log(0) -> 무한대 epsilon 리턴 -> 10-7
    # z가 1-epsilon보다 크면 -log(0) -> 무한대 1-epsilon 리턴 -> 0.9999 리턴
    
    costs = -(y * np.log(z_safe) + (1-y) * np.log(1-z_safe))
    mean_cost = np.mean(costs)
    
    # 100번마다 한 번씩 학습 상태를 출력
    if epoch % 100 == 0 or epoch == epochs - 1:
        print(f'Epoch {epoch}, 평균 비용(Cost): {mean_cost:.6f}, a: {a}, b: {b}')
    print('='*30)

H=[-0.13416408 -0.04472136  0.04472136  0.13416408]
z=[0.4665092  0.48882152 0.51117848 0.5334908 ]
grad_a=-0.42224770144375845
grad_b=2.7755575615628914e-17
update a = 0.14222477014437584, update b = -2.7755575615628915e-18
Epoch 0, 평균 비용(Cost): 0.649675, a: 0.14222477014437584, b: -2.7755575615628915e-18
H=[-0.19081455 -0.06360485  0.06360485  0.19081455]
z=[0.45244058 0.48410415 0.51589585 0.54755942]
grad_a=-0.41175534454951257
grad_b=-2.7755575615628914e-17
update a = 0.1834003045993271, update b = 0.0
H=[-0.24605733 -0.08201911  0.08201911  0.24605733]
z=[0.43879416 0.47950671 0.52049329 0.56120584]
grad_a=-0.4015730318619577
grad_b=0.0
update a = 0.22355760778552286, update b = 0.0
H=[-0.299934 -0.099978  0.099978  0.299934]
z=[0.42557362 0.4750263  0.5249737  0.57442638]
grad_a=-0.39170257011507503
grad_b=0.0
update a = 0.26272786479703036, update b = 0.0
H=[-0.35248642 -0.11749547  0.11749547  0.35248642]
z=[0.4127796  0.47065988 0.52934012 0.5872204 ]
grad_a=-0.38214372240573

In [ ]:
# 학습 완료 후 최종 가중치와 편향 확인

# 학습된 a와 b는 정규화된 입력값 X_norm을 기준으로 학습된 값
print(f'\n학습 완료 후의 최적값: a={a}, b={b}')

In [ ]:
# 새로운 입력값 예측

# 키가 185cm인 사람이 농구선수인지 예측
input_height = 185

# 새로운 입력값도 학습 데이터와 같은 방식으로 정규화해야 함
input_norm = (input_height - X_mean) / X_std

# H(x)를 계산한 뒤, sigmoid를 적용해 확률로 변환
H_input = a * input_norm + b
probability = sigmoid(H_input)

print(f'\n키가 {input_height}cm인 사람이 농구선수일 확률: {probability:.4f}')

# 이진 분류에서는 보통 확률이 0.5 이상이면 1, 0.5 미만이면 0으로 판단
if probability >= 0.5:
    print('판별 결과: 농구선수입니다.')
else:
    print('판별 결과: 농구선수가 아닙니다.')